# Temporal logic

So far we have used assertions and cover statements to verify that boolean conditions hold or are reachable. In this section we will see how to describe properties that stretch over multiple clock cycles.

In [1]:
# basic setup of jupyter notebook

from __future__ import annotations

from example_util.jupyter_util import display_vcd

import cohdl
from cohdl import Port, Signal, Bit, BitVector, Unsigned, std

# When an exception occurs during compilation,
# cohdl inserts fake stack frames into the exception traceback.
# This does not work properly inside jupyter notebooks.
cohdl.use_pretty_traceback(False)

Again, we will use a simple example design. This time a Counter entity that increments its output value on every clock cycle if an enable signal is set to true.

In [2]:
class Counter(cohdl.Entity):
    clk = Port.input(Bit)

    enable = Port.input(Bit)
    result = Port.output(Unsigned[8], default=0)

    def architecture(self):

        @std.sequential(std.Clock(self.clk))
        def gen_counter():
            if self.enable:
                self.result <<= self.result + 1

To reduce code duplication in the rest of this file, we define a wrapper function that receives the example code and runs the verification step for us.

In [3]:
from cohdl_yosys import YosysTestCase, YosysParams, formal

def run_example(*, entity=Counter, display="(*.)clk|enable|result", top_only=True, **kwargs):

    def wrapper(fn):

        class VerifyWrapper(YosysTestCase, entity=entity):
            _yosys_params_ = YosysParams(clean_build_dir=True, quiet=True, **kwargs)

            def architecture(self, dut):
                formal.set_default_ctx(clk=std.Clock(dut.clk))

                @std.concurrent
                def formal_properties():
                    # tested function is called here,
                    # everything else is just common wrapper code
                    fn(dut)

        if VerifyWrapper().test_formal_properties(return_on_error=True):
            print("formal check has passed")
        else:
            print("formal check has failed")

        display_vcd("build/**/*.vcd", display, top_only=top_only)
    
    return wrapper

We will use the cover statement to visualize our examples. As we have already seen in a previous section, the verification tool will try to find some input that satisfies the given condition. Once a solution is found, Yosys exports it as a .vcd file.

In [4]:
from cohdl_yosys.formal import cover

@run_example(cover=True)
def formal_properties(dut: Counter):
    # find a way to produce the value 4 on the result port
    cover[:](dut.result == 4)

formal check has passed
build/project_cover/engine_0/trace0.vcd


## sequences

The `seq` function takes an arbitrary number of boolean arguments. It represents a sequence of clock cycles in which all elements evaluate to true. As seen in the example, the coverage tool manages to produce the sequence 2-2-3 by pulling the enable signal low for one tick.

In [5]:
from cohdl_yosys.formal import seq

@run_example(cover=True)
def formal_properties(dut: Counter):
    res = dut.result

    cover[:](seq(res == 2, res == 2, res == 3))

formal check has passed
build/project_cover/engine_0/trace0.vcd


The arguments of the sequence function can by any kind of boolean expression. They do not have to refer to the same signals or relate to each other in any way. As long as the arguments evaluate to true in consecutive clock cycles, the sequence match is fulfilled. One straight forward consequence of this is, that constants introduce a delay.

In the example we see this because the waveform continues for three more cycles after the value `2` is reached for the first time.

In [6]:
@run_example(cover=True)
def formal_properties(dut: Counter):
    res = dut.result

    cover[:](seq(res == 1, True, True, True, res == 2))

formal check has passed
build/project_cover/engine_0/trace0.vcd


## wait expressions

Since expressing delays is a common task, cohdl_yosys provides the `wait` object. This is more flexible than the approach with hard coded constants because it can also represent ranges of allowed durations. 

```python
wait[3]    # exactly 3 clock cycles
wait[2:5]  # 2,3,4 or 5 clock cycles
wait[:3]   # 0,1,2 or 3 clock cycles
wait[4:]   # 4 for or more clock cycles
wait[:]    # any number of clock cycles
```

In [7]:
from cohdl_yosys.formal import wait

@run_example(cover=True)
def formal_properties(dut: Counter):
    res = dut.result

    cover[:](seq(res == 1, wait[1], res == 2, res == 3))

    # use slices to represent allowed wait durations
    # only lower bound shows up in the output because
    # the tool just chooses the first match
    cover[:](seq(res == 1, wait[2:3], res == 2, res == 3))
    cover[:](seq(res == 1, wait[3:5], res == 2, res == 3))

formal check has passed
build/project_cover/engine_0/trace0.vcd


build/project_cover/engine_0/trace1.vcd


build/project_cover/engine_0/trace2.vcd


## repeat expressions

Sometimes we want to wait but still impose a restriction on the system. As with the hard coded True-True-True sequence we could do this by repeating a condition multiple times. The `repeat` object automates this. It takes a boolean expression as its argument and forces it to be true for consecutive clock cycles. `repeat` supports the same slice operators as `wait`.

`wait[A:B]` is equivalent to `repeat[A:B](True)`.

In [8]:
from cohdl_yosys.formal import repeat

@run_example(cover=True)
def formal_properties(dut: Counter):
    res = dut.result
    en = dut.enable

    cover[:](seq(repeat[1](not en), res == 2)) # seq(not en, res == 2)
    cover[:](seq(repeat[2](not en), res == 2)) # seq(not en, not en, res == 2)
    cover[:](seq(repeat[3](not en), res == 2)) # seq(not en, not en, not en, res == 2)

formal check has passed
build/project_cover/engine_0/trace0.vcd


build/project_cover/engine_0/trace1.vcd


build/project_cover/engine_0/trace2.vcd


## accessing previous state

In our counter example we will eventually want to check that the output value increases. To do that we need a way to access the previous state of the signal. `cohdl_yosys.formal` provides the `prev` function which does just that. Since there are a few caveats when using it, we will first develop our own version to see what issues arise and how they can be fixed.

Generating the previous value of a signal is equivalent to delaying it by one clock cycle. Since we can use arbitrary CoHDL code in the formal architecture method, the easiest way to do that is to assign a new signal in a clocked context. The result is then compared to the value `3` in a cover statement. When I first implemented a version of this function, I expected the output to be a sequence where the second to last digit is `3` (either `0-1-2-3-3` or `0-1-2-3-4`). Yosys however claims that the condition can be fulfilled in the first clock cycle.

In [9]:
@cohdl.pyeval
def get_previous_result(dut: Counter):

    prev_value = Signal[Unsigned[8]]()

    @std.sequential(std.Clock(dut.clk))
    def proc():
        # assign result of entity to a signal
        # to delay it by one clock cycle
        prev_value.next = dut.result
    
    return prev_value

@run_example(cover=True, display="*.clk|*.enable|*.result|*.prev_value", top_only=False)
def formal_properties(dut: Counter):
    cover[:](get_previous_result(dut) == 3)

formal check has passed
build/project_cover/engine_0/trace0.vcd


Ok, so what went wrong? 

With the cover statement we essentially ask the question *'Is it possible that the signal prev_value is equal to 3?'*. Since `prev_value` is first assigned at the first rising edge of the clock, its value is undefined before that event. Since the value is undefined, it could be in any state including `3`.

There are two ways to fix this problem:

* The `past_valid` function returns False during the first clock cycle. We can use it to make the condition False while its argument is not yet defined.
* We can define a default value for the signal. This will prevent the tools from making one up.

In [10]:
from cohdl_yosys.formal import past_valid

@run_example(cover=True)
def formal_properties(dut: Counter):
    
    cover[:](past_valid() and get_previous_result(dut) == 3)

formal check has passed
build/project_cover/engine_0/trace0.vcd


In [11]:
@cohdl.pyeval
def get_previous_result(dut: Counter):

    # set default value to 0
    prev_value = Signal[Unsigned[8]](0)

    @std.sequential(std.Clock(dut.clk))
    def proc():
        prev_value.next = dut.result
    
    return prev_value

@run_example(cover=True)
def formal_properties(dut: Counter):
    cover[:](get_previous_result(dut) == 3)

formal check has passed
build/project_cover/engine_0/trace0.vcd


The cohdl_yosys `prev` function performs essentially the same operation as `get_previous_result` in the above example. It returns the previous state of the argument expression. `prev` has the same limitation of not being able to return a defined value before the first time step. The solutions we saw before also apply here. An optional `default` argument exists to initialize the internal values.

Both `past_valid` and `prev` can look multiple steps into the past.

In [12]:
from cohdl_yosys.formal import prev

@run_example(cover=True)
def formal_properties(dut: Counter):
    res = dut.result

    cover[:](past_valid() and prev(res) == 3)

    cover[:](prev(res, default=0) == 4)

    # past_valid and prev have optional arguments
    # to go back more than one step into the past
    cover[:](past_valid(2) and prev(res, 2) == 4)

    # the default-value solutions also works
    # when going back multiple steps
    cover[:](prev(res, 3, default=0) == 4)

formal check has passed
build/project_cover/engine_0/trace0.vcd


build/project_cover/engine_0/trace1.vcd


build/project_cover/engine_0/trace2.vcd


build/project_cover/engine_0/trace3.vcd


Based on `prev`, cohdl_yosys defines some more functions to describe changes of values:

* `rose(x)`

    Defined for Bit and boolean values. Shorthand for `not prev(x) and x`.

* `fell(x)`

    Defined for Bit and boolean values. Shorthand for `prev(x) and not x`.

* `stable(x)`

    Defined for all types that support equality comparison. Shorthand for `x == prev(x)`.

In [13]:
from cohdl_yosys.formal import rose, fell, stable

@run_example(cover=True)
def formal_properties(dut: Counter):
    res = dut.result
    en = dut.enable

    cover[:](
        seq(
            res == 1,       # start sequence at 1
            rose(en),       # enable changed from low to high
            fell(en),       # enable changed from high to low
            res == 2,
            stable(res),    # result must NOT change in this cycle
            not stable(res) # result must change in this cycle
        )
    )

formal check has passed
build/project_cover/engine_0/trace0.vcd


## When expressions

Properties of digial designs can often be described with statements of the form *'when A then B'*.

examples:

* WHEN a request is received THEN a response must be generated
* WHEN a reset has occurred THEN the entity must be in a defined state

The `When` class is used to express such dependencies. `When(a).then_next(b)` reads as *'When condition a is true, b must be true in the following clock cycle'*.

`When` is also useful to express logical relations in a single clock cycle (immediately). `When(a).then_imm(b)` reads as *'whenever a is true, b is also true'* or simply *'a implies b'*. This is equivalent to `(not a) or b` but usually easier to interpret for humans.

```python
# the following assertions are equivalent:
always[:](When(a).then_imm(b))
always[:](not a or b)
never[:](a and not b)

```

When-expressions can be combined with sequences. Alternatively, method chaining can be used to extend them over more clock cycles.

```python
always[:](When(a).then_next(seq(b,c,d)))
# equivalent to previous line
always[:](When(a).then_next(b).then_next(c).then_next(d))

```

In [14]:
from cohdl_yosys.formal import always, When

@run_example(bmc=True)
def formal_properties(dut: Counter):
    res = dut.result

    # this assertion passes
    always[:](When(dut.enable).then_next(res == prev(res)+1))

    # this assertion passes because it is not possible
    # for the value to be 0 after an increment
    # (overflow is not detected because bmc_depth is to small)
    always[:](When(dut.enable).then_next(res!=0))

    # this assertion fails because it is possible
    # for the value to be 5 after an increment
    always[:](When(dut.enable).then_next(res!=5))

formal check has failed
build/project_bmc/engine_0/trace.vcd


# parallel sequences

The `parallel` function takes an arbitrary number of sequences as arguments. The resulting expression matches, if all of them match at the same time. All parallel branches must cover the same number of clock cycles. You can add elastic `wait[:]` expressions to dynamically adjust the length of sequences.

In [15]:
class DualCounter(cohdl.Entity):
    clk = Port.input(Bit)

    en_a = Port.input(Bit)
    en_b = Port.input(Bit)

    out_a = Port.output(Unsigned[8], default=0)
    out_b = Port.output(Unsigned[8], default=0)

    def architecture(self):

        @std.sequential(std.Clock(self.clk))
        def gen_counter():
            if self.en_a:
                self.out_a <<= self.out_a + 1
            
            if self.en_b:
                self.out_b <<= self.out_b + 1

In [16]:
from cohdl_yosys.formal import parallel


@run_example(entity=DualCounter, cover=True, display="(*.)clk|en_a|en_b|out_a|out_b")
def formal_properties(dut: DualCounter):

    # matches if out_a 1-2-3 and out_b 2-2-3
    cover[:](
        parallel(
            seq(dut.out_a == 1, dut.out_a == 2, dut.out_a == 3),
            seq(dut.out_b == 2, dut.out_b == 2, dut.out_b == 3),
        )
    )

    # matches if out_a 2-3-4 while out_b stable at 3
    cover[:](
        parallel(
            seq(dut.out_a == 2, dut.out_a == 3, dut.out_a == 4),
            repeat[:](dut.out_b == 3),
        )
    )

formal check has passed
build/project_cover/engine_0/trace0.vcd


build/project_cover/engine_0/trace1.vcd
